In [1]:
import pandas as pd
import numpy as np



In [2]:
# Load data
train_data = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/train.csv")
test_data = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/test.csv")


# Simple text‑length feature engineering
def add_length_features(df):
    df = df.copy()
    df["prompt_len"] = df["prompt"].astype(str).apply(len)
    df["response_a_len"] = df["response_a"].astype(str).apply(len)
    df["response_b_len"] = df["response_b"].astype(str).apply(len)
    return df


train_feat = add_length_features(train_data)
test_feat = add_length_features(test_data)

# Features and targets
X_train = train_feat[["prompt_len", "response_a_len", "response_b_len"]]
y_train = train_feat[["winner_model_a", "winner_model_b", "winner_tie"]]
X_test = test_feat[["prompt_len", "response_a_len", "response_b_len"]]



In [3]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
raw_preds = model.predict(X_test)



In [4]:
# Convert raw predictions to probabilities with softmax
exp_preds = np.exp(raw_preds - np.max(raw_preds, axis=1, keepdims=True))
probs = exp_preds / exp_preds.sum(axis=1, keepdims=True)

submission = pd.DataFrame(
    {
        "id": test_data["id"],
        "winner_model_a": probs[:, 0],
        "winner_model_b": probs[:, 1],
        "winner_tie": probs[:, 2],
    }
)



In [5]:
print(submission.head())



           id  winner_model_a  winner_model_b  winner_tie
0  3297560222        0.345633        0.332314    0.322052
1  2556155375        0.231412        0.473335    0.295253
2  1793939629        0.331163        0.337848    0.330989
3  2562993561        0.304957        0.368398    0.326645
4  2111345399        0.354353        0.325580    0.320067


In [6]:
submission.to_csv("submission.csv", index=False)